## Learning about what is Data Skew?
**数据倾斜（Data Skew）** 在分布式重工业管道中的物理本质与设计原罪：

分布式计算的核心精神是“按劳分配、齐头并进”。我们开辟了 100 个分区（Partitions），就是雇佣了 100 个分布式小兵。在理想宇宙里，每个小兵应该各自挑 10 斤砖，大家同时干完，打卡下班。

但当你写下 groupBy("seller_id") 或 join 时，Spark 底层的 Hash 传送门会无情地宣判：【相同的 Key，必须雷打不动地飞向同一个小兵的内存】。

而在真实世界里，数据天生是不公平的。如果大厂官方旗舰店（seller_id = 8888）一天狂砍 500 万单，而其余 9999 个小商贩一天各只有 1 单。Hash 传送门会把这 500 万单全部砸给 1 号小兵，其余小兵手里只有 1 单。
结局就是：99 个小兵用了 0.001 秒就干完收工在原地抽烟，而 1 号小兵在全网的瞩目下，一个人痛苦地挑着 500 万斤的数据独自轰鸣，直到把内存撑爆（OOM）或者把全公司的业务管道生生卡死几小时。

这就叫数据倾斜。不干掉倾斜，你买再多的机器、加再大的算力，都只是在给那些“原地抽烟的偷懒小兵”发工资，根本无法解决那 1% 累死的小兵。


In [0]:
from pyspark.sql import functions as F

In [0]:

#先引入表
oi = spark.table("bronze_order_items")


In [0]:
print("====== 🔍 Analyzing the data distribution of seller_id ======")

seller_dist = oi.groupby("seller_id").agg(F.count("*").alias("row_count"))

seller_stats = seller_dist.agg(
    F.max("row_count").alias("max"),
    F.min("row_count").alias("min"),
    F.mean("row_count").alias("mean"),
    F.stddev("row_count").alias("stddev")
).collect()[0]

max_s, min_s,mean_s, std_s = seller_stats["max"],seller_stats["min"], seller_stats["mean"], seller_stats["stddev"]

skew_ratio_s = max_s / mean_s if mean_s > 0 else 0

print(f"【最大单 Key 记录数】: {max_s} 行")
print(f"【全局平均每 Key 记录数】: {mean_s:.2f} 行")
print(f"【标准差（数据波动剧烈度）】: {std_s:.2f}")
print(f"【倾斜倍率（最大/平均）】: {skew_ratio_s:.2f} 倍")

一般来说，Skew_ratio:
```
= 1  是完全无倾斜
> 5  业务需要关注
> 10 明显倾斜
> 20 严重倾斜
```

由此可见，在olist中，seller存在严重的数据倾斜情况

In [0]:
print("====== 🔍 Analyzing the data distribution of product_id ======")

product_dist = oi.groupby("product_id").agg(F.count("*").alias("row_count"))

product_stats = product_dist.agg(
    F.max("row_count").alias("max"),
    F.min("row_count").alias("min"),
    F.mean("row_count").alias("mean"),
    F.stddev("row_count").alias("stddev")
).collect()[0]

max_p, min_p,mean_p, std_p = product_stats["max"],product_stats["min"], product_stats["mean"], product_stats["stddev"]

skew_ratio_p = max_p / mean_p if mean_p > 0 else 0

print(f"【最大单 Key 记录数】: {max_p} 行")
print(f"【全局平均每 Key 记录数】: {mean_p:.2f} 行")
print(f"【标准差（数据波动剧烈度）】: {std_p:.2f}")
print(f"【倾斜倍率（最大/平均）】: {skew_ratio_p:.2f} 倍")

product也存在严重的数据倾斜情况

In [0]:
print("\n👉 贡献流量最大的 Top 5 超级卖家（显眼包）：")
seller_dist.orderBy(F.col("row_count").desc()).show(5)


print("\n" + "="*60 + "\n")

# 重工业大数据管道：数据倾斜（Data Skew）体检与手术标记白皮书

## 概念物理落盘
* **数据倾斜定义**：由于业务具有天然的【长尾效应】（如头部卖家、大促爆款），导致相同 Key 的巨量数据通过 Hash 门时聚集在单台机器内存中，摧毁了分布式的绝对并行性。

---

## 调优手术路径标记清单（Marked Tables for Optimization）

根据宏观指标与 Top 5 显眼包的对账结果，为下一阶段的数据管道施加如下物理隔离：

* [ ] **标记 1：[seller_id 隔离层]** ➔ 如果确诊倾斜，对该字段标记 **【两阶段加盐聚合（Salting Aggregation）】** 手术，强行打碎头部大卖家的物理聚集。
* [ ] **标记 2：[product_id 隔离层]** ➔ 如果发现爆款单 Key 行数极大，将该超级商品 ID 剥离为维表，施加 **【Broadcast Join 广播空投】** 手术，使其绕过 Shuffle 下水道。
